In [20]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
import requests
import os
import pandas as pd
import gc
import pickle
import joblib
import seaborn as sns

# Mice data

---
## supervised mouse experiments before training
VR2_2021_03_20_1 original example mouse  
TX60_2021_04_10_1  
TX108_2023_03_13_1  

## supervised mouse experiments after training
VR2_2021_04_06_1 original example mouse  
TX60_2021_05_04_1   
TX108_2023_03_22_1  

---
## unsupervised mouse experiments before training
TX105_2022_10_08_2 original example mouse   
TX123_2023_12_21_1  
TX83_2022_08_17_1  

## unsupervised mouse experiments after training
TX105_2022_10_19_2 original example mouse  
TX123_2024_01_02_1  
TX83_2022_08_29_1  


# Downloading the data

In [21]:
# @title Set path for data
root = r'/content/Zhong_et_al_2025' # TODO: set your path here, replace `/content/Zhong_et_al_2025` with your preferred directory path
Path(root).mkdir(exist_ok=True)

In [42]:
# @title Data loading and imports

# download files from Figshare
file_ID = [54183860, 54183863, 54183917, 54183914, 54184790, 54184673, 54184637, 54184628, 54184787, 54184700, 54184742, 54184481, 54184592, 54184583, 54184670, 54184679]

'''
File ID mappings are: (These are from https://janelia.figshare.com/articles/dataset/Unsupervised_pretraining_in_biological_neural_network/28811129/2)
54183860: Beh_sup_train1_after_learning.npy
54183863: Beh_sup_train1_before_learning.npy
54183917: Beh_unsup_train1_after_learning.npy
54183914: Beh_unsup_train1_before_learning.npy
54184742: TX105_2022_10_08_2_neural_data.npy
54184583: TX105_2022_10_19_2_neural_data.npy
54184787: TX108_2023_03_13_1_neural_data.npy
54184700: TX108_2023_03_22_1_neural_data.npy
54184481: TX123_2023_12_21_1_neural_data.npy
54184670: TX123_2024_01_02_1_neural_data.npy
54184637: TX60_2021_04_10_1_neural_data.npy
54184628: TX60_2021_05_04_1_neural_data.npy
54184592: TX83_2022_08_17_1_neural_data.npy
54184679: TX83_2022_08_29_1_neural_data.npy
54184790: VR2_2021_03_20_1_neural_data.npy
54184673: VR2_2021_04_06_1_neural_data.npy
'''

BASE_URL = 'https://api.figshare.com/v2'
r = requests.get(BASE_URL + '/articles/' + str(28811129)) # 28811129 is the ID of the whole dataset
file_metadata = json.loads(r.text)
for file in file_metadata['files']:
  if file['id'] in file_ID: # only download files included in file_ID
    fn = os.path.join(root, file['name'])
    if not os.path.isfile(fn):
      response = requests.get(BASE_URL + '/file/download/' + str(file['id']))
      open(fn, 'wb').write(response.content)
    print(fn)

/content/Zhong_et_al_2025/Beh_sup_train1_after_learning.npy
/content/Zhong_et_al_2025/Beh_sup_train1_before_learning.npy
/content/Zhong_et_al_2025/Beh_unsup_train1_after_learning.npy
/content/Zhong_et_al_2025/Beh_unsup_train1_before_learning.npy
/content/Zhong_et_al_2025/TX105_2022_10_08_2_neural_data.npy
/content/Zhong_et_al_2025/TX105_2022_10_19_2_neural_data.npy
/content/Zhong_et_al_2025/TX108_2023_03_13_1_neural_data.npy
/content/Zhong_et_al_2025/TX108_2023_03_22_1_neural_data.npy
/content/Zhong_et_al_2025/TX123_2023_12_21_1_neural_data.npy
/content/Zhong_et_al_2025/TX123_2024_01_02_1_neural_data.npy
/content/Zhong_et_al_2025/TX60_2021_04_10_1_neural_data.npy
/content/Zhong_et_al_2025/TX60_2021_05_04_1_neural_data.npy
/content/Zhong_et_al_2025/TX83_2022_08_17_1_neural_data.npy
/content/Zhong_et_al_2025/TX83_2022_08_29_1_neural_data.npy
/content/Zhong_et_al_2025/VR2_2021_03_20_1_neural_data.npy
/content/Zhong_et_al_2025/VR2_2021_04_06_1_neural_data.npy


# Load data

In [25]:
# @title Setting target ids

sup_bef_1 = "VR2_2021_03_20_1"
sup_bef_2 = "TX60_2021_04_10_1"
sup_bef_3 = "TX108_2023_03_13_1"

sup_aft_1 = "VR2_2021_04_06_1"
sup_aft_2 = "TX60_2021_05_04_1"
sup_aft_3 = "TX108_2023_03_22_1"

unsup_bef_1 = "TX105_2022_10_08_2"
unsup_bef_2 = "TX123_2023_12_21_1"
unsup_bef_3 = "TX83_2022_08_17_1"

unsup_aft_1 = "TX105_2022_10_19_2"
unsup_aft_2 = "TX123_2024_01_02_1"
unsup_aft_3 = "TX83_2022_08_29_1 "

In [26]:
# @title Setting target files
target_file = sup_bef_1 # Change this as needed

target_index = int(target_file[-1])

if target_file[:-2] == "sup_bef":
  target_beh_filename = "Beh_sup_train1_before_learning.npy"
elif target_file[:-2] == "sup_aft":
  target_beh_filename = "Beh_sup_train1_after_learning.npy"
elif target_file[:-2] == "unsup_bef":
  target_beh_filename = "Beh_unsup_train1_before_learning.npy"
elif target_file[:-2] == "unsup_aft":
  target_beh_filename = "Beh_unsup_train1_after_learning.npy"

In [30]:
# @title Load spiking data
spiking_raw_data = np.load(os.path.join(root, target_file + '_neural_data.npy'), allow_pickle=True)

UnpicklingError: Failed to interpret file '/content/Zhong_et_al_2025/VR2_2021_03_20_1_neural_data.npy' as a pickle

In [ ]:
# @title Load behavioral data

beh = np.load(os.path.join(root, target_path), allow_pickle=1).item()[target_file]

# Data search (These cells are for exploration only, no need to run them)

In [18]:
raw_dict = np.load("Imaging_Exp_info.npy",  allow_pickle=True).item()

all_sessions = []

for phase_name, sessions in raw_dict.items():
    # 'sessions' is an array of dictionaries
    for session in sessions:
        # Create a copy to avoid modifying original data
        entry = session.copy()

        # Add the phase name (the dictionary key) as a column
        entry['phase'] = phase_name

        # Clean up specific fields for readability
        if 'stim_id' in entry and isinstance(entry['stim_id'], np.ndarray):
            entry['stim_id'] = str(entry['stim_id'].tolist())

        if 'depth' in entry and isinstance(entry['depth'], list):
            entry['depth'] = f"{entry['depth'][0]}, {entry['depth'][1]}"

        all_sessions.append(entry)

# Convert to DataFrame
df = pd.DataFrame(all_sessions)

# Reorder columns to put most important info first
cols = ['phase', 'mname', 'datexp', 'exptype', 'rewType', 'Gender', 'Note']
remaining_cols = [c for c in df.columns if c not in cols]
df = df[cols + remaining_cols]

# Sort by phase and mouse name for clarity
df = df.sort_values(['phase', 'mname'])

# Display the result
df.iloc[0, :].T

,43
phase,naive_test1
mname,LZ13
datexp,2024_05_15
exptype,NaN
rewType,None
Gender,Female
Note,VR move when run
sess#,1.0
blk,1
is2p,1


In [19]:
# 1. Define your target ID list
target_ids = [
    'VR2_2021_03_20_1', 'TX60_2021_04_10_1', 'TX108_2023_03_13_1', # Supervised Before
    'VR2_2021_04_06_1', 'TX60_2021_05_04_1', 'TX108_2023_03_22_1', # Supervised After
    'TX105_2022_10_08_2', 'TX123_2023_12_21_1', 'TX83_2022_08_17_1', # Unsupervised Before
    'TX105_2022_10_19_2', 'TX123_2024_01_02_1', 'TX83_2022_08_29_1'  # Unsupervised After
]

# 2. Create a temporary 'full_id' column to match your format
# Note: datexp in your array is YYYY_MM_DD, but your list uses underscores.
# We ensure the mapping is clean by joining the columns.
df['temp_id'] = (
    df['mname'].astype(str) + '_' +
    df['datexp'].astype(str) + '_' +
    df['blk'].astype(str)
)

# 3. Filter the dataframe
extracted_df = df[df['temp_id'].isin(target_ids)].copy()

# 4. Clean up (remove the temporary ID column and reset index)
extracted_df = extracted_df.drop(columns=['temp_id']).reset_index(drop=True)

# 5. Display result
print(f"Extracted {len(extracted_df)} rows.")
print(extracted_df[['phase', 'mname', 'datexp', 'blk']])

Extracted 12 rows.
                           phase  mname      datexp blk
0      sup_train1_after_learning  TX108  2023_03_22   1
1      sup_train1_after_learning   TX60  2021_05_04   1
2      sup_train1_after_learning    VR2  2021_04_06   1
3     sup_train1_before_learning  TX108  2023_03_13   1
4     sup_train1_before_learning   TX60  2021_04_10   1
5     sup_train1_before_learning    VR2  2021_03_20   1
6    unsup_train1_after_learning  TX105  2022_10_19   2
7    unsup_train1_after_learning  TX123  2024_01_02   1
8    unsup_train1_after_learning   TX83  2022_08_29   1
9   unsup_train1_before_learning  TX105  2022_10_08   2
10  unsup_train1_before_learning  TX123  2023_12_21   1
11  unsup_train1_before_learning   TX83  2022_08_17   1
